# 03 — Evaluation: метрики и figures для ВКР (v4 noleak edition)

Источник чисел — `docs/thesis/data_methodology.md` §14. Финальная конфигурация:
- модели `*_v4_mpnet_tfidf_noleak` (`scripts/build_noleak_artifacts.py`);
- gold: LLM-consensus (3257 cells, primary) + HUMAN/Opus (615 cells, conservative bound);
- router: `category_router_v5`;
- эталонные скрипты:
  - `python -m src.eval.metric_table` → `datasets/processed/v4_metric_table_v2.json`
  - `python -m src.eval.end_to_end`    → `datasets/processed/v4_e2e_router_eval.json`

Запускать рекомендуется на VM (`/home/miafrolov/Desktop/diploma`), где доступен `off_work/{cat}_off_full.parquet`.
На локальной машине, если артефактов нет, ноутбук всё равно выведет числа методички §14 (placeholder mode) и сохранит 3 PNG.

**Регенерирует 3 figures в `images/`:** `tier_breakdown.png` (per-attribute macro-F1 bar), `cost_quality_scatter.png`, `layer_contribution.png`.

**Дефолт:** light, без heavy шагов. Включай `HEAVY['regenerate_metrics']=True` чтобы пересчитать json'ы через subprocess.

In [1]:
%env OMP_NUM_THREADS=1

import sys
from pathlib import Path

assert sys.version_info[:2] in [(3, 12), (3, 14)]

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / 'datasets' / 'processed'
MODELS = PROJECT_ROOT / 'models'
IMAGES = PROJECT_ROOT / 'images'
IMAGES.mkdir(exist_ok=True)

CATEGORIES_LONG = ['pasta', 'chocolate', 'cheeses']

# Heavy steps:
#  cascade_predict      — legacy pre-v4 pipeline (cascade_preds_*_v2_gold_hybrid_v3_fixed.parquet)
#  regenerate_metrics   — `python -m src.eval.metric_table` + `src.eval.end_to_end`
#                         требует доступ к off_full parquet (обычно VM)
HEAVY = {
    'cascade_predict':    False,
    'regenerate_metrics': False,
}

import matplotlib
matplotlib.use('Agg')  # детерминированный backend
import matplotlib.pyplot as plt
import numpy as np
np.random.seed(42)
plt.rcParams['savefig.dpi'] = 150

import pandas as pd
import json

SAVEFIG_KW = dict(dpi=150, bbox_inches='tight',
                  metadata={'Date': None, 'Software': None, 'Creator': None})

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

env: OMP_NUM_THREADS=1


PROJECT_ROOT = /Users/miafrolov/Desktop/stuff/ai_attributes


In [2]:
import seaborn as sns
import matplotlib.ticker as mticker

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['legend.fontsize'] = 10

PALETTE = sns.color_palette('viridis', 8)
TAB10 = sns.color_palette('tab10', 10)
print(f'PROCESSED exists: {PROCESSED.is_dir()}')
print(f'MODELS exists:    {MODELS.is_dir()}')
print(f'IMAGES exists:    {IMAGES.is_dir()}')

PROCESSED exists: True
MODELS exists:    True
IMAGES exists:    True


## 0. Heavy regenerate (опционально)

Если `HEAVY['regenerate_metrics']=True`, ноутбук запустит `src.eval.metric_table` и `src.eval.end_to_end` через subprocess. Требуется доступ к `off_work/{cat}_off_full.parquet` (`/home/miafrolov/off_work/` на VM) и `models/{cat}_v4_mpnet_tfidf_noleak_*`.

Output: `datasets/processed/v4_metric_table_v2.json` и `datasets/processed/v4_e2e_router_eval.json`.

Legacy шаг `cascade_predict` оставлен только для секций 6+ (Bayes validator, ECE и пр.).

In [3]:
import subprocess

if HEAVY['regenerate_metrics']:
    for mod in ['src.eval.metric_table', 'src.eval.end_to_end']:
        try:
            print(f'>>> python -m {mod}')
            r = subprocess.run(['python', '-m', mod], cwd=PROJECT_ROOT,
                               check=False, timeout=1800,
                               capture_output=True, text=True)
            print(r.stdout[-1500:])
            if r.returncode != 0:
                print('STDERR:', r.stderr[-800:])
        except Exception as e:
            print(f'{mod} skipped: {e!r}')

if HEAVY['cascade_predict']:
    for cat in CATEGORIES_LONG:
        out = PROCESSED / f'cascade_preds_{cat}_v2_gold_hybrid_v3_fixed.parquet'
        if not out.exists():
            try:
                subprocess.run(
                    ['python', '-m', 'src.eval.cascade_predict', '--category', cat],
                    cwd=PROJECT_ROOT, check=False, timeout=600,
                )
            except Exception as e:
                print(f'{cat} cascade_predict skipped: {e}')
        print(f'{cat}: {"OK" if out.exists() else "missing"} {out.name}')

metric_table_path = PROCESSED / 'v4_metric_table_v2.json'
e2e_router_path = PROCESSED / 'v4_e2e_router_eval.json'
print()
print(f'v4_metric_table_v2.json:  {"OK" if metric_table_path.exists() else "MISSING"}')
print(f'v4_e2e_router_eval.json:  {"OK" if e2e_router_path.exists() else "MISSING"}')
if not metric_table_path.exists() or not e2e_router_path.exists():
    print('Если артефактов нет — секции 1–4 будут использовать hard-coded числа из methodology §14.')
    print('Для регенерации: на VM HEAVY["regenerate_metrics"]=True ИЛИ ручной запуск:')
    print('  python -m src.eval.metric_table && python -m src.eval.end_to_end')


v4_metric_table_v2.json:  MISSING
v4_e2e_router_eval.json:  MISSING
Если артефактов нет — секции 1–4 будут использовать hard-coded числа из methodology §14.
Для регенерации: на VM HEAVY["regenerate_metrics"]=True ИЛИ ручной запуск:
  python -m src.eval.metric_table && python -m src.eval.end_to_end


## 1. Headline на LLM-consensus gold (n=3257) — primary

Источник: `src.eval.end_to_end` + `src.eval.metric_table` на `manual_gold_consensus.parquet` (3-LLM majority vote, n=3257 cells × {pasta, chocolate, cheeses}).

Целевые числа (methodology §14.1):
- Cascade-only micro: **94.8 %**, macro-F1 (cells-weighted) **0.899**;
- E2E (None=wrong, production-realistic): **91.1 %**, coverage **96.0 %**;
- Router accuracy (per code, n=570): **95.4 %**.

In [4]:
# Reference numbers from methodology §14.1
HEADLINE_CONSENSUS = {
    'n_cells':              3257,
    'cascade_only_micro':   0.948,
    'cascade_only_macro_f1_cells_weighted': 0.899,
    'cascade_only_macro_f1_attr_unweighted': 0.902,
    'e2e_acc_none_as_wrong': 0.911,
    'e2e_coverage':          0.960,
    'router_acc':            0.954,
    'router_n_codes':        570,
}

def _load_json(p):
    if not p.exists():
        return None
    with open(p, encoding='utf-8') as f:
        return json.load(f)

mt = _load_json(metric_table_path)
e2e = _load_json(e2e_router_path)

def _cells_weighted(attrs_dict, key):
    ns = [v['n'] for v in attrs_dict.values()]
    vals = [v[key] for v in attrs_dict.values()]
    return float(np.average(vals, weights=ns)), int(sum(ns))

if mt and 'LLM-consensus' in mt:
    attrs = mt['LLM-consensus']
    micro_w, n_total = _cells_weighted(attrs, 'micro_acc')
    macro_w, _ = _cells_weighted(attrs, 'macro_f1')
    print(f'[v4 metric_table] LLM-consensus: n_cells={n_total}, micro={micro_w*100:.1f}%, macro-F1={macro_w:.3f}')
else:
    print('WARNING: v4_metric_table_v2.json отсутствует — fallback на methodology §14.1')

if e2e and 'LLM-consensus' in e2e:
    r = e2e['LLM-consensus']
    print(f'[v4 end_to_end] LLM-consensus: cascade_only={r["cascade_only_acc"]*100:.1f}%, '
          f'E2E(None=wrong)={r["e2e_acc_none_as_wrong"]*100:.1f}%, '
          f'coverage={r["e2e_coverage"]*100:.1f}%, router={r["router_acc"]*100:.1f}%')
else:
    print('WARNING: v4_e2e_router_eval.json отсутствует — fallback на methodology §14.1')

print()
print('=' * 78)
print('Headline (LLM-consensus gold, n=3257) — methodology §14.1:')
print('=' * 78)
rows = pd.DataFrame([
    {'metric': 'Cascade-only micro-accuracy',           'value': f"{HEADLINE_CONSENSUS['cascade_only_micro']*100:.1f}%"},
    {'metric': 'Cascade-only macro-F1 (cells-weighted)', 'value': f"{HEADLINE_CONSENSUS['cascade_only_macro_f1_cells_weighted']:.3f}"},
    {'metric': 'Cascade-only macro-F1 (attr-unweighted)','value': f"{HEADLINE_CONSENSUS['cascade_only_macro_f1_attr_unweighted']:.3f}"},
    {'metric': 'Router accuracy (per code, n=570)',     'value': f"{HEADLINE_CONSENSUS['router_acc']*100:.1f}%"},
    {'metric': 'E2E coverage',                          'value': f"{HEADLINE_CONSENSUS['e2e_coverage']*100:.1f}%"},
    {'metric': 'E2E accuracy (None=wrong) — production', 'value': f"{HEADLINE_CONSENSUS['e2e_acc_none_as_wrong']*100:.1f}%"},
])
display(rows)


Headline (LLM-consensus gold, n=3257) — methodology §14.1:


,metric,value
0,Cascade-only micro-accuracy,94.8%
1,Cascade-only macro-F1 (cells-weighted),0.899
2,Cascade-only macro-F1 (attr-unweighted),0.902
3,"Router accuracy (per code, n=570)",95.4%
4,E2E coverage,96.0%
5,E2E accuracy (None=wrong) — production,91.1%


## 2. Conservative bound — HUMAN gold (Opus, n=615)

Источник: `manual_eval_per_product.parquet` (single labeler Opus, без IRR). Используется как нижняя оценка (conservative bound).

Целевые числа (methodology §14.2):
- Cascade-only: **91.7 %**, macro-F1 **0.848**;
- E2E (None=wrong): **87.5 %**, coverage **95.4 %**;
- Router (n=107): **95.3 %**.

In [5]:
HEADLINE_HUMAN = {
    'n_cells':              615,
    'cascade_only_micro':   0.917,
    'cascade_only_macro_f1': 0.848,
    'e2e_acc_none_as_wrong': 0.875,
    'e2e_coverage':          0.954,
    'router_acc':            0.953,
    'router_n_codes':        107,
}

if mt and 'HUMAN' in mt:
    attrs = mt['HUMAN']
    micro_w, n_total = _cells_weighted(attrs, 'micro_acc')
    macro_w, _ = _cells_weighted(attrs, 'macro_f1')
    print(f'[v4 metric_table] HUMAN: n_cells={n_total}, micro={micro_w*100:.1f}%, macro-F1={macro_w:.3f}')
else:
    print('WARNING: HUMAN data в v4_metric_table_v2.json отсутствует — fallback на §14.2')

if e2e and 'HUMAN' in e2e:
    r = e2e['HUMAN']
    print(f'[v4 end_to_end] HUMAN: cascade_only={r["cascade_only_acc"]*100:.1f}%, '
          f'E2E(None=wrong)={r["e2e_acc_none_as_wrong"]*100:.1f}%, '
          f'coverage={r["e2e_coverage"]*100:.1f}%, router={r["router_acc"]*100:.1f}%')
else:
    print('WARNING: HUMAN data в v4_e2e_router_eval.json отсутствует — fallback на §14.2')

print()
print('=' * 78)
print('Conservative bound (HUMAN gold, Opus, n=615) — methodology §14.2:')
print('=' * 78)
rows = pd.DataFrame([
    {'metric': 'Cascade-only micro-accuracy', 'value': f"{HEADLINE_HUMAN['cascade_only_micro']*100:.1f}%"},
    {'metric': 'Cascade-only macro-F1',        'value': f"{HEADLINE_HUMAN['cascade_only_macro_f1']:.3f}"},
    {'metric': 'Router accuracy (n=107)',      'value': f"{HEADLINE_HUMAN['router_acc']*100:.1f}%"},
    {'metric': 'E2E coverage',                 'value': f"{HEADLINE_HUMAN['e2e_coverage']*100:.1f}%"},
    {'metric': 'E2E accuracy (None=wrong)',    'value': f"{HEADLINE_HUMAN['e2e_acc_none_as_wrong']*100:.1f}%"},
])
display(rows)


Conservative bound (HUMAN gold, Opus, n=615) — methodology §14.2:


,metric,value
0,Cascade-only micro-accuracy,91.7%
1,Cascade-only macro-F1,0.848
2,Router accuracy (n=107),95.3%
3,E2E coverage,95.4%
4,E2E accuracy (None=wrong),87.5%


## 3. Category router accuracy (Layer 0)

`category_router_v5` (XGBoost на MPNet embeddings) — pre-cascade Layer 0. Эталонные числа:
- LLM-consensus codes (n=570): **95.4 %**;
- HUMAN codes (n=107): **95.3 %**.

Источник: `src.eval.end_to_end` → `v4_e2e_router_eval.json`.

In [6]:
router_rows = pd.DataFrame([
    {'gold': 'LLM-consensus', 'n_codes': HEADLINE_CONSENSUS['router_n_codes'],
     'router_acc_%': HEADLINE_CONSENSUS['router_acc']*100},
    {'gold': 'HUMAN (Opus)',  'n_codes': HEADLINE_HUMAN['router_n_codes'],
     'router_acc_%': HEADLINE_HUMAN['router_acc']*100},
])
if e2e:
    for gl, r in e2e.items():
        router_rows.loc[router_rows.gold.str.startswith(gl), 'router_acc_%'] = float(r['router_acc']) * 100
display(router_rows.round(2))

,gold,n_codes,router_acc_%
0,LLM-consensus,570,95.4
1,HUMAN (Opus),107,95.3


## 4. Per-attribute macro-F1 (LLM-consensus gold, 20 атрибутов)

Methodology §14.3 — 20 строк: 7 pasta + 6 chocolate + 7 cheeses. Источник: `src.eval.metric_table`.

**Регенерирует `images/tier_breakdown.png`** — bar chart per-attribute macro-F1 (адаптирован под v4 schema).

In [7]:
# Methodology §14.3 — per-attribute table
PER_ATTR_TABLE = [
    # (category, attr, n, micro_acc, macro_f1)
    ('pasta',     'grain_type',         139, 0.942, 0.807),
    ('pasta',     'pasta_shape',        139, 0.993, 0.984),
    ('pasta',     'is_filled',          210, 0.957, 0.793),
    ('pasta',     'is_gluten_free',     194, 0.943, 0.942),
    ('pasta',     'is_organic',         193, 0.974, 0.969),
    ('pasta',     'is_vegan',           200, 0.955, 0.939),
    ('pasta',     'cuisine_origin',     198, 0.909, 0.790),
    ('chocolate', 'chocolate_type',     150, 0.980, 0.973),
    ('chocolate', 'is_filled',          198, 0.949, 0.861),
    ('chocolate', 'chocolate_extra',    157, 0.911, 0.899),
    ('chocolate', 'contains_nuts',      172, 0.890, 0.888),
    ('chocolate', 'is_organic',         169, 0.982, 0.967),
    ('chocolate', 'flavor_profile',     136, 0.949, 0.898),
    ('cheeses',   'milk_source',        147, 0.973, 0.972),
    ('cheeses',   'texture',            130, 0.923, 0.927),
    ('cheeses',   'country_of_origin',  125, 0.952, 0.901),
    ('cheeses',   'aging',              119, 0.933, 0.889),
    ('cheeses',   'is_pdo',             140, 0.979, 0.939),
    ('cheeses',   'is_organic',         158, 0.987, 0.967),
    ('cheeses',   'is_ultra_processed', 154, 0.961, 0.920),
]
per_attr = pd.DataFrame(PER_ATTR_TABLE,
                        columns=['category', 'attr', 'n', 'micro', 'macro_f1'])

# Если есть v4_metric_table_v2.json — переписываем значения из реального прогона
source_label = 'methodology §14.3 (hard-coded)'
if mt and 'LLM-consensus' in mt:
    attrs = mt['LLM-consensus']
    rebuilt = []
    for cat, attr, n_fallback, micro_fb, macro_fb in PER_ATTR_TABLE:
        key = f'{cat}.{attr}'
        if key in attrs:
            a = attrs[key]
            rebuilt.append((cat, attr, a['n'], a['micro_acc'], a['macro_f1']))
        else:
            rebuilt.append((cat, attr, n_fallback, micro_fb, macro_fb))
    per_attr = pd.DataFrame(rebuilt, columns=['category', 'attr', 'n', 'micro', 'macro_f1'])
    source_label = 'v4_metric_table_v2.json (live)'

per_attr_disp = per_attr.copy()
per_attr_disp['micro'] = (per_attr_disp['micro'] * 100).round(1).astype(str) + '%'
per_attr_disp['macro_f1'] = per_attr_disp['macro_f1'].round(3)
print(f'Источник: {source_label}')
display(per_attr_disp)

# === Figure: per-attribute macro-F1 (replaces tier_breakdown) ===
tier_fig_path = IMAGES / 'tier_breakdown.png'
fig, ax = plt.subplots(figsize=(11, 7.2))
cat_colors = {'pasta': PALETTE[1], 'chocolate': PALETTE[3], 'cheeses': PALETTE[6]}
per_attr_plot = per_attr.sort_values(['category', 'macro_f1'], ascending=[True, True]).reset_index(drop=True)
y = np.arange(len(per_attr_plot))
colors = [cat_colors[c] for c in per_attr_plot['category']]
bars = ax.barh(y, per_attr_plot['macro_f1'].values, color=colors,
               edgecolor='black', linewidth=0.4)
for bar, v, n in zip(bars, per_attr_plot['macro_f1'], per_attr_plot['n']):
    ax.text(v + 0.004, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}  (n={int(n)})', va='center', fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels([f"{r.category[:3]}/{r.attr}" for r in per_attr_plot.itertuples()])
ax.set_xlim(0.6, 1.06)
ax.set_xlabel('macro-F1 (cells-weighted внутри атрибута)')
ax.set_title('Per-attribute macro-F1 каскада (v4 noleak) на LLM-consensus gold (n=3257)')
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=cat_colors[c], label=c) for c in ['pasta', 'chocolate', 'cheeses']]
ax.legend(handles=legend_handles, loc='lower right', title='категория')
plt.tight_layout()
fig.savefig(tier_fig_path, **SAVEFIG_KW)
plt.close(fig)
print(f'saved: {tier_fig_path}')

# Сводные средние
cells = per_attr['n'].sum()
cw_micro = np.average(per_attr['micro'], weights=per_attr['n'])
cw_macro = np.average(per_attr['macro_f1'], weights=per_attr['n'])
print(f'\nCells-weighted: micro={cw_micro*100:.1f}%, macro-F1={cw_macro:.3f} '
      f'(reference §14.1: 94.8% / 0.899)')
print(f'Attr-unweighted macro-F1 = {per_attr["macro_f1"].mean():.3f} (reference: 0.902)')

Источник: methodology §14.3 (hard-coded)


,category,attr,n,micro,macro_f1
0,pasta,grain_type,139,94.2%,0.807
1,pasta,pasta_shape,139,99.3%,0.984
2,pasta,is_filled,210,95.7%,0.793
3,pasta,is_gluten_free,194,94.3%,0.942
4,pasta,is_organic,193,97.4%,0.969
5,pasta,is_vegan,200,95.5%,0.939
6,pasta,cuisine_origin,198,90.9%,0.790
7,chocolate,chocolate_type,150,98.0%,0.973
8,chocolate,is_filled,198,94.9%,0.861
9,chocolate,chocolate_extra,157,91.1%,0.899


saved: /Users/miafrolov/Desktop/stuff/ai_attributes/images/tier_breakdown.png

Cells-weighted: micro=95.2%, macro-F1=0.909 (reference §14.1: 94.8% / 0.899)
Attr-unweighted macro-F1 = 0.911 (reference: 0.902)


## 5. LLM fallback distribution (per-layer contribution, n=3506 eval cells)

Methodology §14.4 — распределение источников финальной метки по 4 слоям:
- Layer 1 (rule_h): **18.4 %** (646 cells)
- Layer 2 (ML: MPNet + TF-IDF SVD + XGBoost): **73.8 %** (2588 cells) — главная работа
- Layer 3 (rule_l): 0.7 % (23 cells)
- Layer 4 (LLM fallback): **7.1 %** (249 cells)

**LLM cost reduction vs all-LLM baseline: 92.9 %**.

Per-category: pasta 10.9 %, chocolate 2.0 %, cheeses 6.9 %.

**Регенерирует `images/layer_contribution.png`.**

In [8]:
LAYER_DIST = pd.DataFrame([
    {'layer': 'Layer 1\n(rule_h)',      'cells': 646,  'pct': 18.4},
    {'layer': 'Layer 2\n(ML)',          'cells': 2588, 'pct': 73.8},
    {'layer': 'Layer 3\n(rule_l)',      'cells': 23,   'pct': 0.7},
    {'layer': 'Layer 4\n(LLM fallback)','cells': 249,  'pct': 7.1},
])
PER_CAT_LLM = pd.DataFrame([
    {'category': 'pasta',     'llm_fallback_pct': 10.9},
    {'category': 'chocolate', 'llm_fallback_pct': 2.0},
    {'category': 'cheeses',   'llm_fallback_pct': 6.9},
])
display(LAYER_DIST)
display(PER_CAT_LLM)

layer_fig_path = IMAGES / 'layer_contribution.png'
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))

# Left: stacked bar by layer (overall)
ax = axes[0]
LAYER_COLORS = [PALETTE[6], PALETTE[3], PALETTE[2], PALETTE[1]]
bars = ax.bar(LAYER_DIST['layer'], LAYER_DIST['cells'],
              color=LAYER_COLORS, edgecolor='black', linewidth=0.5)
for b, pct, n in zip(bars, LAYER_DIST['pct'], LAYER_DIST['cells']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 30,
            f'{pct:.1f}%\n(n={int(n)})', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Число ячеек (final source)')
ax.set_ylim(0, max(LAYER_DIST['cells']) * 1.18)
ax.set_title('Источник финальной метки по слоям каскада (всего n=3506)')
ax.text(0.02, 0.95,
        f'LLM cost reduction\nvs naive all-LLM: 92.9 %',
        transform=ax.transAxes, ha='left', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff8e1',
                  edgecolor='#aa8800', linewidth=0.6, alpha=0.95))

# Right: per-category LLM fallback share
ax = axes[1]
cat_colors = {'pasta': PALETTE[1], 'chocolate': PALETTE[3], 'cheeses': PALETTE[6]}
colors = [cat_colors[c] for c in PER_CAT_LLM['category']]
bars = ax.bar(PER_CAT_LLM['category'], PER_CAT_LLM['llm_fallback_pct'],
              color=colors, edgecolor='black', linewidth=0.5)
for b, v in zip(bars, PER_CAT_LLM['llm_fallback_pct']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
            f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Доля Layer 4 (LLM fallback), %')
ax.set_ylim(0, 14)
ax.set_title('LLM fallback rate по категориям')

fig.suptitle('Распределение работы каскада (v4 noleak) по слоям',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(layer_fig_path, **SAVEFIG_KW)
plt.close(fig)
print(f'saved: {layer_fig_path}')

# Опционально: если есть legacy cascade_preds_after_fix.parquet — выводим для сравнения
try:
    rows = []
    for cat in ['pasta', 'chocolate', 'cheeses']:
        df = pd.read_parquet(PROCESSED / f'cascade_preds_{cat}_after_fix.parquet')
        vc = df['layer'].value_counts(normalize=True) * 100
        rows.append({'category': cat, **vc.to_dict()})
    legacy_layers = pd.DataFrame(rows).fillna(0).round(1)
    print('\n[legacy reference, pre-v4]')
    display(legacy_layers)
except Exception as e:
    print(f'\nlegacy layer distribution unavailable: {e!r}')

,layer,cells,pct
0,Layer 1\n(rule_h),646,18.4
1,Layer 2\n(ML),2588,73.8
2,Layer 3\n(rule_l),23,0.7
3,Layer 4\n(LLM fallback),249,7.1


,category,llm_fallback_pct
0,pasta,10.9
1,chocolate,2.0
2,cheeses,6.9


saved: /Users/miafrolov/Desktop/stuff/ai_attributes/images/layer_contribution.png



[legacy reference, pre-v4]


,category,ml,regex,abstain
0,pasta,73.8,18.0,8.2
1,chocolate,71.9,22.7,5.4
2,cheeses,93.7,2.8,3.5


## 6. Cost-quality matrix (legacy reference)

Источник: `cascade_plus_llm4_summary.parquet` + `cascade_plus_llm4_hybrid.parquet` — legacy pre-v4 артефакты.
Для финального headline используем числа methodology §14 (см. секции 1–2).
Эта секция демонстрирует Pareto-структуру 11 точек cascade_only / all_llm / cascade+LLM.

**Регенерирует `images/cost_quality_scatter.png`.**

In [9]:
COST_REL = {
    'sonnet45':      1.000,
    'gpt4o':         0.727,
    'gemini25flash': 0.042,
    'gptoss':        0.030,
    'llama3b':       0.009,
}
COST_CASCADE_EPS_PLOT = 0.0005

cost_fig_path = IMAGES / 'cost_quality_scatter.png'
try:
    summary = pd.read_parquet(PROCESSED / 'cascade_plus_llm4_summary.parquet')
    summary = summary.sort_values('grand_acc_hybrid_with_router', ascending=False).reset_index(drop=True)
    hybrid = pd.read_parquet(PROCESSED / 'cascade_plus_llm4_hybrid.parquet')
    all_llm_acc_per_model = (hybrid.groupby('llm_model')
                                    .apply(lambda g: (g['llm_acc_on_attr'] * g['n_test']).sum()
                                                      / g['n_test'].sum())
                                    .to_dict())
    cascade_only_acc = summary['grand_acc_cascade_only_e2e'].iloc[0]
    coverage = summary['grand_coverage'].iloc[0]

    points = [{'label': 'cascade_only', 'model': '-', 'kind': 'cascade',
               'cost': COST_CASCADE_EPS_PLOT, 'acc': cascade_only_acc}]
    for _, r in summary.iterrows():
        m = r['llm_model']
        if m not in COST_REL:
            continue
        points.append({'label': f'all_llm/{m}', 'model': m, 'kind': 'all_llm',
                       'cost': COST_REL[m],
                       'acc': all_llm_acc_per_model.get(m, float('nan'))})
        points.append({'label': f'cascade+{m}', 'model': m, 'kind': 'hybrid',
                       'cost': (1 - coverage) * COST_REL[m],
                       'acc': r['grand_acc_hybrid_with_router']})
    pts = pd.DataFrame(points)

    fig, ax = plt.subplots(figsize=(14, 8))
    colors = {'cascade': PALETTE[1], 'all_llm': PALETTE[6], 'hybrid': PALETTE[3]}
    markers = {'cascade': 's', 'all_llm': '^', 'hybrid': 'o'}
    kind_labels = {'cascade': 'каскад', 'all_llm': 'прямой LLM', 'hybrid': 'гибрид'}

    for kind, grp in pts.groupby('kind'):
        ax.scatter(grp['cost'], grp['acc'] * 100, s=160,
                   color=colors[kind], marker=markers[kind],
                   edgecolor='black', linewidth=0.6,
                   label=kind_labels.get(kind, kind), zorder=3)

    pts_sorted = pts.sort_values('cost').reset_index(drop=True)
    frontier_x, frontier_y, best_acc = [], [], -1.0
    for _, r in pts_sorted.iterrows():
        if r['acc'] > best_acc:
            frontier_x.append(r['cost'])
            frontier_y.append(r['acc'] * 100)
            best_acc = r['acc']
    ax.plot(frontier_x, frontier_y, '--', color='darkred', alpha=0.55,
            linewidth=1.4, label='парето-граница', zorder=2)

    label_offsets = {
        'cascade+gemini25flash': (10, 12),
        'cascade+gptoss':        (10, -22),
        'cascade+sonnet45':      (-110, 8),
        'all_llm/gpt4o':         (-90, 10),
        'all_llm/sonnet45':      (40, 10),
        'all_llm/llama3b':       (10, 6),
        'all_llm/gemini25flash': (-110, -4),
        'cascade_only':          (-110, -10),
        'cascade+gpt4o':         (10, 4),
        'cascade+llama3b':       (15, 14),
        'all_llm/gptoss':        (10, -4),
    }
    winners = ['cascade+gemini25flash', 'cascade+gptoss']
    for _, r in pts.iterrows():
        lbl = r['label']
        dx, dy = label_offsets.get(lbl, (8, 6))
        is_winner = lbl in winners
        ax.annotate(lbl, (r['cost'], r['acc'] * 100),
                    xytext=(dx, dy), textcoords='offset points',
                    fontsize=10 if is_winner else 8.5,
                    fontweight='bold' if is_winner else 'normal',
                    color='darkred' if is_winner else '#333333',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                              edgecolor='none', alpha=0.78))
    ax.set_xscale('log')
    ax.set_xlabel('Относительная стоимость инференса (log scale, sonnet45 ≡ 1.0)')
    ax.set_ylabel('Точность на gold-датасете, %')
    ax.set_title('Cost-quality matrix (legacy gold, 11 рабочих точек, парето-граница пунктиром)')
    xmin, xmax = pts['cost'].min(), pts['cost'].max()
    ax.set_xlim(xmin * 0.4, xmax * 1.8)
    ax.set_ylim(50, 95)
    ax.legend(title='Тип конфигурации', loc='lower left',
              bbox_to_anchor=(1.0, 0.0), borderaxespad=0)
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout(rect=[0, 0, 0.88, 1])
    fig.savefig(cost_fig_path, **SAVEFIG_KW)
    plt.close(fig)
    print(f'saved: {cost_fig_path}')
    display(pts.sort_values('cost').round(4))
except Exception as e:
    print(f'WARNING: cost-quality scatter failed: {e!r}')
    print('Запусти legacy cascade_plus_llm4 чтобы регенерировать parquet, либо игнорируй.')
    # Placeholder из methodology §14.4 — упрощённая схема
    fig, ax = plt.subplots(figsize=(10, 6))
    ref_points = pd.DataFrame([
        {'label': 'cascade_only',  'cost': 0.0005, 'acc': 94.8, 'kind': 'cascade'},
        {'label': 'cascade+LLM',   'cost': 0.071,  'acc': 91.1, 'kind': 'hybrid'},
        {'label': 'all_llm/sonnet45', 'cost': 1.000, 'acc': 83.8, 'kind': 'all_llm'},
    ])
    colors = {'cascade': PALETTE[1], 'all_llm': PALETTE[6], 'hybrid': PALETTE[3]}
    for _, r in ref_points.iterrows():
        ax.scatter(r['cost'], r['acc'], s=200, color=colors[r['kind']],
                   edgecolor='black', linewidth=0.6, zorder=3)
        ax.annotate(r['label'], (r['cost'], r['acc']),
                    xytext=(10, 8), textcoords='offset points', fontsize=10)
    ax.set_xscale('log')
    ax.set_xlabel('Относительная стоимость инференса (log scale, sonnet45 ≡ 1.0)')
    ax.set_ylabel('Точность, %')
    ax.set_title('Cost-quality (placeholder, methodology §14)')
    ax.set_ylim(70, 100)
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    fig.savefig(cost_fig_path, **SAVEFIG_KW)
    plt.close(fig)
    print(f'placeholder saved: {cost_fig_path}')

saved: /Users/miafrolov/Desktop/stuff/ai_attributes/images/cost_quality_scatter.png


,label,model,kind,cost,acc
0,cascade_only,-,cascade,0.0005,0.8713
10,cascade+llama3b,llama3b,hybrid,0.0008,0.8797
8,cascade+gptoss,gptoss,hybrid,0.0026,0.9055
2,cascade+gemini25flash,gemini25flash,hybrid,0.0036,0.9152
9,all_llm/llama3b,llama3b,all_llm,0.0090,0.5619
7,all_llm/gptoss,gptoss,all_llm,0.0300,0.6934
1,all_llm/gemini25flash,gemini25flash,all_llm,0.0420,0.8290
6,cascade+gpt4o,gpt4o,hybrid,0.0623,0.9107
4,cascade+sonnet45,sonnet45,hybrid,0.0857,0.9134
5,all_llm/gpt4o,gpt4o,all_llm,0.7270,0.8405


## 7. ECE calibration (legacy reference)

Изотоническая калибровка на silver hold-out. Источник: `ece_calibration_table.parquet` (legacy pre-v4). Графика остаётся для иллюстративных целей.

In [10]:
try:
    ece = pd.read_parquet(PROCESSED / 'ece_calibration_table.parquet')
    ece_plot = ece.dropna(subset=['ece_after']).copy()
    if 'n_calib' in ece_plot.columns:
        ece_plot = ece_plot[ece_plot['n_calib'] >= 80]
    ece_plot['key'] = ece_plot['category'].str[:3] + '/' + ece_plot['attr']
    ece_plot = ece_plot.sort_values('ece_before', ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(ece_plot))
    w = 0.4
    ax.barh(x - w/2, ece_plot['ece_before'], w, label='до калибровки',
            color=PALETTE[1], edgecolor='black', linewidth=0.4)
    ax.barh(x + w/2, ece_plot['ece_after'], w, label='после калибровки (isotonic)',
            color=PALETTE[6], edgecolor='black', linewidth=0.4)
    ax.set_yticks(x)
    ax.set_yticklabels(ece_plot['key'])
    ax.invert_yaxis()
    ax.set_xlabel('ECE (expected calibration error)')
    ax.set_title('Калибровка ML-слоя (legacy reference): ECE до и после isotonic regression')
    ax.legend()
    plt.tight_layout()
    plt.close(fig)
    print('ECE calibration: legacy parquet processed (figure not saved to images/).')
except Exception as e:
    print(f'WARNING: ece calibration unavailable: {e!r}')

ECE calibration: legacy parquet processed (figure not saved to images/).


## 8. Bayes-валидатор (legacy reference)

В финальной конфигурации v4 Bayes-валидатор НЕ используется (см. MEMORY.md, accuracy_squeeze_deploy_2026-05-20). Секция оставлена для иллюстрации эксперимента.

In [11]:
try:
    bv = pd.read_parquet(PROCESSED / 'bayes_validator_demote_metric.parquet').copy()
    bv['tp'] = bv['n_flag_cascade_wrong']
    bv['fp'] = bv['n_flag_cascade_right']
    bv_per_cat = bv.groupby('category', as_index=False)[['tp', 'fp', 'tn', 'fn']].sum()
    display(bv_per_cat)

    ex = bv[(bv['category'] == 'chocolate') & (bv['attr'] == 'contains_nuts')]
    if len(ex):
        ex_disp = ex[['category', 'attr', 'n_flagged', 'flag_rate',
                      'demote_precision', 'demote_recall',
                      'demote_precision_lift', 'expected_delta_acc_if_demote']].copy()
        for c in ['flag_rate', 'demote_precision', 'demote_recall', 'demote_precision_lift',
                  'expected_delta_acc_if_demote']:
            ex_disp[c] = ex_disp[c].round(4)
        print('Пример работающего атрибута — chocolate/contains_nuts:')
        display(ex_disp)
except Exception as e:
    print(f'WARNING: bayes validator legacy parquet unavailable: {e!r}')

,category,tp,fp,tn,fn
0,cheeses,1,87,1296,52
1,chocolate,22,87,644,19
2,pasta,2,78,918,12


Пример работающего атрибута — chocolate/contains_nuts:


,category,attr,n_flagged,flag_rate,demote_precision,demote_recall,demote_precision_lift,expected_delta_acc_if_demote
11,chocolate,contains_nuts,41,0.1715,0.5366,0.5946,0.3818,0.0665


## 9. H1 negative — обучаемый router (legacy)

Гипотеза H1: XGBoost-router способен предсказывать передачу в LLM-fallback лучше static_threshold. На brand-disjoint pre-registered eval гипотеза **отклонена**: Δ(router − static) ≈ 0.45 п.п. при медианной ширине CI 5.14 п.п.

In [12]:
try:
    rp = pd.read_parquet(PROCESSED / 'router_pareto_gold.parquet')
    def _closest(strategy, target_cost):
        g = rp[rp['strategy'] == strategy].copy()
        g['d'] = (g['cost'] - target_cost).abs()
        return g.nsmallest(1, 'd')
    target = 0.20
    cmp = pd.concat([
        _closest('router', target),
        _closest('static_threshold', target),
        _closest('random', target),
    ]).reset_index(drop=True)
    print('Парето-сравнение в точке cost ≈ 0.20:')
    print(cmp[['strategy', 'threshold', 'cost', 'accuracy']].round(4).to_string(index=False))
    delta_pp = (cmp[cmp.strategy == 'router']['accuracy'].iloc[0]
                - cmp[cmp.strategy == 'static_threshold']['accuracy'].iloc[0]) * 100
    print(f'\nΔ(router − static_threshold) в точке cost ≈ 0.20: {delta_pp:+.2f} п.п.')
    print('Медианная ширина 95% brand-clustered CI на атрибут: ~5.14 п.п.')
    print('Вывод: |Δ| меньше медианной ширины CI ⇒ статистически неотличимы. H1 отклонена.')
except Exception as e:
    print(f'WARNING: H1 negative router parquet unavailable: {e!r}')
    print('Reference: Δ ≈ +0.45 п.п., медианный 95% CI ≈ 5.14 п.п. (см. memory/router_h1_decision.md).')

Парето-сравнение в точке cost ≈ 0.20:
        strategy  threshold   cost  accuracy
          router       0.60 0.2008    0.8220
static_threshold       0.75 0.2216    0.8174
          random       0.20 0.2151    0.7862

Δ(router − static_threshold) в точке cost ≈ 0.20: +0.45 п.п.
Медианная ширина 95% brand-clustered CI на атрибут: ~5.14 п.п.
Вывод: |Δ| меньше медианной ширины CI ⇒ статистически неотличимы. H1 отклонена.


## 10. Chapter 4 summary — достижения (methodology §14)

Сводная таблица «повышено / ускорено / сокращено» — итоговые числа для §4.3 ВКР.

In [13]:
ch4 = pd.DataFrame([
    {'параметр': 'Точность каскада (LLM-consensus gold, micro)',
     'значение': '94.8 %', 'источник': 'methodology §14.1'},
    {'параметр': 'Cells-weighted macro-F1 (LLM-consensus)',
     'значение': '0.899', 'источник': 'methodology §14.1'},
    {'параметр': 'E2E accuracy (router → cascade, None=wrong)',
     'значение': '91.1 %', 'источник': 'methodology §14.1'},
    {'параметр': 'E2E coverage',
     'значение': '96.0 %', 'источник': 'methodology §14.1'},
    {'параметр': 'Router accuracy (per code, n=570)',
     'значение': '95.4 %', 'источник': 'methodology §14.1'},
    {'параметр': 'Conservative bound (HUMAN gold, cascade-only)',
     'значение': '91.7 %', 'источник': 'methodology §14.2'},
    {'параметр': 'Conservative bound (HUMAN gold, E2E)',
     'значение': '87.5 %', 'источник': 'methodology §14.2'},
    {'параметр': 'LLM fallback rate (Layer 4)',
     'значение': '7.1 %', 'источник': 'methodology §14.4'},
    {'параметр': 'LLM cost reduction vs naive all-LLM',
     'значение': '92.9 %', 'источник': 'methodology §14.4'},
    {'параметр': 'Доля ML слоя (главная работа)',
     'значение': '73.8 %', 'источник': 'methodology §14.4'},
    {'параметр': 'Доля rule_h слоя',
     'значение': '18.4 %', 'источник': 'methodology §14.4'},
    {'параметр': 'Truly-independent estimate (после учёта circular bias)',
     'значение': '~91 % cascade / ~87 % E2E', 'источник': 'methodology §14.5'},
])
display(ch4)

,параметр,значение,источник
0,"Точность каскада (LLM-consensus gold, micro)",94.8 %,methodology §14.1
1,Cells-weighted macro-F1 (LLM-consensus),0.899,methodology §14.1
2,"E2E accuracy (router → cascade, None=wrong)",91.1 %,methodology §14.1
3,E2E coverage,96.0 %,methodology §14.1
4,"Router accuracy (per code, n=570)",95.4 %,methodology §14.1
5,"Conservative bound (HUMAN gold, cascade-only)",91.7 %,methodology §14.2
6,"Conservative bound (HUMAN gold, E2E)",87.5 %,methodology §14.2
7,LLM fallback rate (Layer 4),7.1 %,methodology §14.4
8,LLM cost reduction vs naive all-LLM,92.9 %,methodology §14.4
9,Доля ML слоя (главная работа),73.8 %,methodology §14.4


**Ключевые цифры главы 4** (methodology §14):

- Headline на LLM-consensus gold (n=3257): cascade-only **94.8 %**, E2E **91.1 %** (None=wrong).
- Conservative bound на HUMAN gold (n=615, Opus): cascade-only **91.7 %**, E2E **87.5 %** — совпадает с truly-independent оценкой §14.5.
- LLM fallback всего **7.1 %** ячеек ⇒ снижение стоимости относительно all-LLM baseline на **92.9 %**.
- Router (Layer 0) — **95.4 %** на 570 кодах consensus / **95.3 %** на 107 кодах human gold.

**Финальная проверка регенерации figures:**

```
images/tier_breakdown.png       (per-attribute macro-F1)
images/cost_quality_scatter.png (Pareto trade-off)
images/layer_contribution.png   (LLM fallback distribution)
```

In [14]:
for fname in ['tier_breakdown.png', 'cost_quality_scatter.png', 'layer_contribution.png']:
    p = IMAGES / fname
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f'OK   {p}  ({size_kb:.1f} KB)')
    else:
        print(f'MISSING  {p}')

OK   /Users/miafrolov/Desktop/stuff/ai_attributes/images/tier_breakdown.png  (142.8 KB)
OK   /Users/miafrolov/Desktop/stuff/ai_attributes/images/cost_quality_scatter.png  (108.5 KB)
OK   /Users/miafrolov/Desktop/stuff/ai_attributes/images/layer_contribution.png  (98.3 KB)
